# Математический маятник: Эйлер vs RK4, энергия, фазовые портреты и «гармонически‑сопряжённый» набор

Рассматриваем математический маятник длины $L$ в поле тяжести $g$.

## Уравнение движения

$$
\ddot\theta + \frac{g}{L}\sin\theta = 0.
$$

Введём фазовые переменные $ \theta $ и $ \omega=\dot\theta $. Тогда

$$
\dot\theta = \omega,\qquad 
\dot\omega = -\frac{g}{L}\sin\theta.
$$

## Энергия

Полная механическая энергия (массу $m$ можно положить равной 1 — она сокращается):

$$
E(\theta,\omega)=\frac12 (L\omega)^2 + gL(1-\cos\theta).
$$

## Малые колебания (гармонический осциллятор)

Для $|\theta|\ll 1$ выполняется $\sin\theta\approx \theta$, и получается линейное уравнение

$$
\ddot\theta + \omega_0^2 \theta = 0,\qquad \omega_0=\sqrt{\frac{g}{L}}.
$$

Его аналитическое решение:
$$
\theta(t)=\theta_0\cos(\omega_0 t)+\frac{\omega^{\text{init}}}{\omega_0}\sin(\omega_0 t),
\quad 
\omega(t)=\dot\theta(t).
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib import animation
from IPython.display import HTML

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.35,
})


## Численные методы

### Явный Эйлер
Для системы $\dot y = f(t,y)$ явный шаг Эйлера:

$$
y_{n+1}=y_n + h f(t_n,y_n).
$$

Это метод 1‑го порядка: глобальная ошибка $\mathcal O(h)$. Для данной задачи даёт **дрейф энергии**.

### RK4 (классический метод Рунге–Кутты 4‑го порядка)

$$
\begin{aligned}
k_1 &= f(t_n,y_n),\\
k_2 &= f(t_n+\tfrac h2, y_n+\tfrac h2 k_1),\\
k_3 &= f(t_n+\tfrac h2, y_n+\tfrac h2 k_2),\\
k_4 &= f(t_n+h, y_n+h k_3),\\
y_{n+1} &= y_n + \tfrac h6 (k_1+2k_2+2k_3+k_4).
\end{aligned}
$$

Глобальная ошибка $\mathcal O(h^4)$; энергия сохраняется существенно лучше, но на очень больших временах дрейф всё равно возможен.


In [ ]:
def pendulum_rhs(t, y, g=9.81, L=1.0):
    '''
    Правая часть для y=[theta, omega]:
        theta_dot = omega
        omega_dot = -(g/L) * sin(theta)
    '''
    theta, omega = y
    return np.array([omega, -(g/L)*np.sin(theta)], dtype=float)

def step_euler(t, y, h, rhs, **rhs_kwargs):
    g = rhs_kwargs.get("g", 9.81)
    L = rhs_kwargs.get("L", 1.0)

    y_next = y
    y_next[1] = y[1] - h * (g/L) * y[0]
    y_next[0] = y[0] + h * y_next[1]
    return y_next

def step_rk4(t, y, h, rhs, **rhs_kwargs):
    k1 = rhs(t, y, **rhs_kwargs)
    k2 = rhs(t + 0.5*h, y + 0.5*h*k1, **rhs_kwargs)
    k3 = rhs(t + 0.5*h, y + 0.5*h*k2, **rhs_kwargs)
    k4 = rhs(t + h, y + h*k3, **rhs_kwargs)
    return y + (h/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def simulate(theta0, omega0, T, h, method="rk4", g=9.81, L=1.0):
    '''
    Интегрирование на [0, T] с шагом h.
    method: "euler" | "rk4"
    Возвращает t, theta, omega.
    '''
    n = int(np.ceil(T/h))
    t = np.linspace(0.0, n*h, n+1)
    y = np.zeros((n+1, 2), dtype=float)
    y[0] = [theta0, omega0]

    step = step_rk4 if method.lower() == "rk4" else step_euler
    for i in range(n):
        y[i+1] = step(t[i], y[i], h, pendulum_rhs, g=g, L=L)
    return t, y[:, 0], y[:, 1]

def energy(theta, omega, g=9.81, L=1.0, m=1.0):
    '''
    E = 1/2 m (L*omega)^2 + m g L (1 - cos(theta))
    '''
    return 0.5*m*(L*omega)**2 + m*g*L*(1 - np.cos(theta))

def analytic_small_angle(t, theta0, omega0, g=9.81, L=1.0):
    '''
    Аналитика для малых углов: theta'' + (g/L) theta = 0.
    '''
    w0 = np.sqrt(g/L)
    theta = theta0*np.cos(w0*t) + (omega0/w0)*np.sin(w0*t)
    omega = -theta0*w0*np.sin(w0*t) + omega0*np.cos(w0*t)
    return theta, omega


## 1) Малые колебания: сравнение с аналитическим решением


In [ ]:
g = 9.81
L = 1.0

theta0 = 0.01   # рад
omega0 = 0.0    # рад/с

w0 = np.sqrt(g/L)
T0 = 2*np.pi/w0
T = 20*T0       # 20 периодов

h = T0/200      # шаг

t_e, th_e, om_e = simulate(theta0, omega0, T=T, h=h, method="euler", g=g, L=L)
t_r, th_r, om_r = simulate(theta0, omega0, T=T, h=h, method="rk4",   g=g, L=L)

th_a, om_a = analytic_small_angle(t_r, theta0, omega0, g=g, L=L)

fig, ax = plt.subplots()
ax.plot(t_e, th_e, label="Эйлер")
ax.plot(t_r, th_r, label="RK4")
ax.plot(t_r, th_a, "--", label="Аналитика (малые углы)")
ax.set_xlabel("t, c")
ax.set_ylabel(r"$\theta(t)$, рад")
ax.set_title("Малые колебания: траектория во времени")
ax.legend()
plt.show()


### Фазовый портрет (малые углы)


In [ ]:
fig, ax = plt.subplots()
ax.plot(th_e, om_e, label="Эйлер")
ax.plot(th_r, om_r, label="RK4")
ax.plot(th_a, om_a, "--", label="Аналитика")
ax.set_xlabel(r"$\theta$, рад")
ax.set_ylabel(r"$\omega$, рад/с")
ax.set_title("Фазовый портрет при малых углах")
ax.legend()
plt.show()


### Энергия и дрейф


In [ ]:
E_e = energy(th_e, om_e, g=g, L=L)
E_r = energy(th_r, om_r, g=g, L=L)
E_a = energy(th_a, om_a, g=g, L=L)

fig, ax = plt.subplots()
ax.plot(t_e, E_e, label="Эйлер")
ax.plot(t_r, E_r, label="RK4")
ax.plot(t_r, E_a, "--", label="Аналитика (линейная модель)")
ax.set_xlabel("t, c")
ax.set_ylabel("E (m=1)")
ax.set_title("Энергия во времени (малые углы)")
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(t_e, (E_e - E_e[0]) / E_e[0], label="Эйлер")
ax.plot(t_r, (E_r - E_r[0]) / E_r[0], label="RK4")
ax.set_xlabel("t, c")
ax.set_ylabel(r"$\frac{E(t)-E(0)}{E(0)}$")
ax.set_title("Относительный дрейф энергии")
ax.legend()
plt.show()


## 2) Накопление ошибки: сравнение с аналитикой


In [ ]:
eps = 1e-16

err_th_e = np.abs(th_e - th_a)
err_th_r = np.abs(th_r - th_a)

err_E_e = np.abs(E_e - E_a)
err_E_r = np.abs(E_r - E_a)

fig, ax = plt.subplots()
ax.semilogy(t_r, err_th_e + eps, label=r"$|\Delta\theta|$ Эйлер")
ax.semilogy(t_r, err_th_r + eps, label=r"$|\Delta\theta|$ RK4")
ax.set_xlabel("t, c")
ax.set_ylabel("ошибка")
ax.set_title(r"Накопление ошибки по $\theta(t)$ (лог. шкала)")
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.semilogy(t_r, err_E_e + eps, label=r"$|\Delta E|$ Эйлер")
ax.semilogy(t_r, err_E_r + eps, label=r"$|\Delta E|$ RK4")
ax.set_xlabel("t, c")
ax.set_ylabel("ошибка энергии")
ax.set_title(r"Накопление ошибки по энергии (лог. шкала)")
ax.legend()
plt.show()

mask = (t_r > 2*T0)  # пропускаем начальную часть
x = np.log(t_r[mask] + eps)

def fit_power_law(err):
    y = np.log(err[mask] + eps)
    a, b = np.polyfit(x, y, 1)
    return a, b

a_th_e, _ = fit_power_law(err_th_e)
a_th_r, _ = fit_power_law(err_th_r)
a_E_e, _  = fit_power_law(err_E_e)
a_E_r, _  = fit_power_law(err_E_r)

print("Оценка наклона a в log(err) ~ a log(t) + b (грубая оценка):")
print(f"  theta: Euler a≈{a_th_e:.2f}, RK4 a≈{a_th_r:.2f}")
print(f"  Energy: Euler a≈{a_E_e:.2f}, RK4 a≈{a_E_r:.2f}")


### Сходимость по шагу $h$ (малые углы)

Сравниваем $\max |\Delta\theta|$ на фиксированном горизонте при нескольких шагах $h$.


In [ ]:
hs = [T0/2, T0/4, T0/8, T0/16, T0/32]
T_test = T0

def max_theta_error_for_h(h, method):
    t, th, om = simulate(theta0, omega0, T=T_test, h=h, method=method, g=g, L=L)
    th_a, _ = analytic_small_angle(t, theta0, omega0, g=g, L=L)
    return np.max(np.abs(th - th_a))

errs_e = [max_theta_error_for_h(hh, "euler") for hh in hs]
errs_r = [max_theta_error_for_h(hh, "rk4") for hh in hs]

fig, ax = plt.subplots()
ax.loglog(hs, errs_e, "o-", label="Эйлер")
ax.loglog(hs, errs_r, "o-", label="RK4")
ax.set_xlabel("шаг h")
ax.set_ylabel(r"$\max_t |\Delta\theta|$ на [0, T_test]")
ax.set_title("Сходимость по шагу")
ax.legend()
plt.show()

p_e, _ = np.polyfit(np.log(hs), np.log(errs_e), 1)
p_r, _ = np.polyfit(np.log(hs), np.log(errs_r), 1)
print(f"Оценка порядка: Euler p≈{p_e:.2f} (ожид. ~1), RK4 p≈{p_r:.2f} (ожид. ~4)")


## 3) Фазовые портреты маятника при разных начальных углах (нелинейная модель)


In [ ]:
h = 0.002
T = 20.0

theta0_list = [0.2, 0.6, 1.0, 1.4, 2.0, 2.6]  # рад
omega0 = 0.0

fig, ax = plt.subplots(figsize=(7, 6))
for th0 in theta0_list:
    t, th, om = simulate(th0, omega0, T=T, h=h, method="rk4", g=g, L=L)
    ax.plot(th, om, label=fr"$\theta_0={th0:.1f}$")
ax.set_xlabel(r"$\theta$, рад")
ax.set_ylabel(r"$\omega$, рад/с")
ax.set_title("Фазовые портреты нелинейного маятника (RK4)")
ax.legend()
plt.show()


## 4) Анимация одного маятника


In [ ]:
L_anim = 1.0
theta0_anim = 0.2
omega0_anim = 0.0
T_anim = 2 * (2*np.pi/np.sqrt(g/L_anim))
h_anim = T_anim / 1000

t, th, om = simulate(theta0_anim, omega0_anim, T=T_anim, h=h_anim, method="rk4", g=g, L=L_anim)

x = L_anim*np.sin(th)
y = -L_anim*np.cos(th)

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_aspect("equal", "box")
ax.set_xlim(-1.2*L_anim, 1.2*L_anim)
ax.set_ylim(-1.2*L_anim, 0.3*L_anim)
ax.set_title("Колеблющийся маятник (RK4)")

line, = ax.plot([], [], lw=2)
bob,  = ax.plot([], [], "o", ms=10)
time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes)

def init():
    line.set_data([], [])
    bob.set_data([], [])
    time_text.set_text("")
    return line, bob, time_text

def animate(i):
    line.set_data([0, x[i]], [0, y[i]])
    bob.set_data([x[i]], [y[i]])
    time_text.set_text(f"t = {t[i]:.2f} c")
    return line, bob, time_text

ani = animation.FuncAnimation(fig, animate, frames=len(t), init_func=init,
                              interval=int(1000*h_anim), blit=True)
plt.close(fig)
HTML(ani.to_jshtml())


## 5) Гармонически‑сопряжённые маятники: рациональные соотношения частот

Задаём $\omega_i = r_i\,\omega_0$ (где $r_i\in\mathbb{Q}$). Тогда
$$
L_i = \frac{L_0}{r_i^2}.
$$


In [ ]:
L0 = 1.0
ratios = [6, 8, 9, 10, 12]   # рациональные коэффициенты частот
lengths = [L0/(r**2) for r in ratios]

theta0_multi = 0.3
omega0_multi = 0.0

T_multi = np.lcm.reduce(ratios)
h_multi = 0.1

sol = []
for L_i in lengths:
    t, th, om = simulate(theta0_multi, omega0_multi, T=T_multi, h=h_multi, method="rk4", g=g, L=L_i)
    sol.append((L_i, th, om))

fig, ax = plt.subplots()
for r, (L_i, th, om) in zip(ratios, sol):
    ax.plot(t, th, label=fr"$r={r}$, $L={L_i:.3f}$ м")
ax.set_xlabel("t, c")
ax.set_ylabel(r"$\theta(t)$, рад")
ax.set_title("Набор маятников с рациональными соотношениями частот")
ax.legend(ncols=2)
plt.show()

th1 = sol[0][1]
th2 = sol[1][1]
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(th1, th2)
ax.set_xlabel(fr"$\theta$(r={ratios[0]})")
ax.set_ylabel(fr"$\theta$(r={ratios[1]})")
ax.set_title("Траектория в пространстве (θ1, θ2) — аналог Лиссажу")
ax.set_aspect("equal", "box")
plt.show()


### Анимация набора маятников


In [ ]:
offsets = 0*np.linspace(-1.6, 1.6, len(lengths))
coords = []
for (L_i, th, om), x0 in zip(sol, offsets):
    x = x0 + L_i*np.sin(th)
    y = -L_i*np.cos(th)
    coords.append((x0, x, y, L_i))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.set_aspect("equal", "box")
ax.set_xlim(offsets.min()-1.2, offsets.max()+1.2)
ax.set_ylim(-max(lengths)-0.3, 0.6)
ax.set_title("Гармонически‑сопряжённый набор маятников (RK4)")

lines, bobs = [], []
for _ in lengths:
    ln, = ax.plot([], [], lw=2)
    bb, = ax.plot([], [], "o", ms=8)
    lines.append(ln)
    bobs.append(bb)

time_text = ax.text(0.02, 0.92, "", transform=ax.transAxes)

def init():
    for ln, bb in zip(lines, bobs):
        ln.set_data([], [])
        bb.set_data([], [])
    time_text.set_text("")
    return (*lines, *bobs, time_text)

def animate(i):
    for k, (x0, x, y, L_i) in enumerate(coords):
        lines[k].set_data([x0, x[i]], [0, y[i]])
        bobs[k].set_data([x[i]], [y[i]])
    time_text.set_text(f"t = {t[i]:.2f} c")
    return (*lines, *bobs, time_text)

ani = animation.FuncAnimation(fig, animate, frames=len(t), init_func=init,
                              interval=int(1000*h_multi), blit=True)
plt.close(fig)
HTML(ani.to_jshtml())
